# Analytics - Single Source of Truth
the purpuse is to get the data from the dp and do some Analytics

In [1]:
try:
    import os
    import pandas as pd
    from sqlalchemy import create_engine, text
    from dotenv import load_dotenv
    from datetime import datetime, timedelta
    import plotly.express as px
    print("Import Success")
except Exception as e:
    print(f"Error Importing: {e}")
    
load_dotenv()

USER = os.getenv('DB_USER')
PASSWORD = os.getenv('DB_PASSWORD')
HOST = os.getenv('DB_HOST')
PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

connection_string = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
try: 
    engine = create_engine(connection_string)
    print(f"Connected to {DB_NAME} - Ready for analysis!")
except Exception as e: 
    print(f"Faild to Create Connection to the db") 

Import Success
Connected to postgres - Ready for analysis!


## Ways To get the data + manipulte
1. Pandas - it download the db to the pc which can take a while
    * using ```df_table = pd.read_sql('table', engine)```
    * display using ```df_table.head()```
2. SQL - Only give me the data i need (very fast)
    * using ```quert = "SELECT * FROM *** JOIN ***" ```
    * display using ```df = pd.read_sql_query(query, engine)```
    * then using    ```display(df.head())```

* **becuase its ```SELECT``` we dont need ```.connect``` method from sqlalchemy**

In [2]:
po_suppliers_query = """
SELECT 
    suppliers.supplier_name,
    purchase_orders.po_id,
    purchase_orders.actual_arrival,
    purchase_orders.expected_arrival
FROM
    purchase_orders
JOIN suppliers ON suppliers.supplier_id = purchase_orders.supplier_id
"""
df_orders = pd.read_sql_query(po_suppliers_query, engine)
display(df_orders.head())
df_orders.info()

,supplier_name,po_id,actual_arrival,expected_arrival
0,TechParts Germany,5001,2026-03-05 12:31:59.508490,2026-02-25 12:31:59.508490
1,AsianLogistics,5002,2026-02-26 12:31:59.508490,2026-02-25 12:31:59.508490
2,TechParts Germany,5003,2026-02-25 12:31:59.508490,2026-02-25 12:31:59.508490
3,IsraelPrecision,5004,NaT,2026-03-12 12:31:59.508490


<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   supplier_name     4 non-null      str           
 1   po_id             4 non-null      int64         
 2   actual_arrival    3 non-null      datetime64[us]
 3   expected_arrival  4 non-null      datetime64[us]
dtypes: datetime64[us](2), int64(1), str(1)
memory usage: 260.0 bytes


### Define Functions To get Manipulated data like I Want

## 1. Suppliers Delays

In [3]:
def analayse_delays(df):
    df_resulte = df.copy()
    df_resulte['delay_days'] = (df_resulte['actual_arrival'] - df_resulte['expected_arrival']).dt.days 
    return df_resulte[df_resulte['delay_days'] > 0]

def get_vendor_delay_report(df):
    late_data = analayse_delays(df)

    vendor_summary = late_data.groupby('supplier_name').agg(
    total_delays=('po_id', 'count'),
    average_delay=('delay_days', 'mean'),
    max_delay=('delay_days', 'max')
    ).reset_index()
    
    return vendor_summary.sort_values(by='average_delay', ascending=False)

late_suppliers_report = analayse_delays(df_orders)
print("Supplier Delay ALL:")
display(late_suppliers_report)

vendor_report = get_vendor_delay_report(df_orders)
print('Avg Suppliers delay by days:')
display(vendor_report)


if not vendor_report.empty:
    worst_vendor = vendor_report.iloc[0]
    print(f"worst supplier by avarage is: {worst_vendor['supplier_name']}")
else:
    print("No delays found right now")

Supplier Delay ALL:


,supplier_name,po_id,actual_arrival,expected_arrival,delay_days
0,TechParts Germany,5001,2026-03-05 12:31:59.508490,2026-02-25 12:31:59.508490,8.0
1,AsianLogistics,5002,2026-02-26 12:31:59.508490,2026-02-25 12:31:59.508490,1.0


Avg Suppliers delay by days:


,supplier_name,total_delays,average_delay,max_delay
1,TechParts Germany,1,8.0,8.0
0,AsianLogistics,1,1.0,1.0


worst supplier by avarage is: TechParts Germany


## 2. Top Selling Items

In [4]:
deliveries_query = """
SELECT 
    customer_deliveries.customer_name,
    parts.part_name,
    parts.category,
    customer_deliveries.quantity,
    customer_deliveries.delivery_date,
    customer_deliveries.status
FROM
    customer_deliveries
JOIN parts ON customer_deliveries.part_id = parts.part_id
"""
df_deliveries = pd.read_sql_query(deliveries_query, engine)
display(df_deliveries.head())
df_deliveries.info()

,customer_name,part_name,category,quantity,delivery_date,status
0,Intel,Motherboard V2,Electronics,150,2026-03-06 12:31:59.508490,Shipped
1,Tesla,Steel Frame,Structure,50,2026-03-17 12:31:59.508490,Pending
2,Apple,Motherboard V2,Electronics,200,2026-03-05 12:31:59.508490,Delivered
3,SpaceX,Hydraulic Pump,Mechanical,30,2026-03-12 12:31:59.508490,Shipped


<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_name  4 non-null      str           
 1   part_name      4 non-null      str           
 2   category       4 non-null      str           
 3   quantity       4 non-null      int64         
 4   delivery_date  4 non-null      datetime64[us]
 5   status         4 non-null      str           
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 324.0 bytes


In [5]:
def get_top_selling_items_all_time(df):
    top_items = df.groupby(['part_name', 'category'])['quantity'].sum().reset_index()
    return top_items.sort_values(by='quantity', ascending=False)

def get_top_selling_items_30_days(df):
    filterd_dates = datetime.now() - timedelta(days=30)
    resulte_filter = df.loc[df['delivery_date'] > filterd_dates].copy()
    resulte = resulte_filter.groupby(['part_name', 'category'])['quantity'].sum().reset_index()
    return resulte.sort_values(by='quantity', ascending=False)

top_selling_report = get_top_selling_items_all_time(df_deliveries)
print("Top Selling Items (All Times):")
display(top_selling_report)

top_selling_report_30_days = get_top_selling_items_30_days(df_deliveries)
print("Top Selling Items (30 days):")
display(top_selling_report_30_days)

Top Selling Items (All Times):


,part_name,category,quantity
1,Motherboard V2,Electronics,350
2,Steel Frame,Structure,50
0,Hydraulic Pump,Mechanical,30


Top Selling Items (30 days):


,part_name,category,quantity
1,Motherboard V2,Electronics,350
2,Steel Frame,Structure,50
0,Hydraulic Pump,Mechanical,30


## Supply Chain Analysis - Vendor Delays

In [12]:
fig_delays = px.bar(
    vendor_report,
    x = 'supplier_name',
    y = 'average_delay',
    title='Avg Delay days per supplier',
    labels={'supplier_name': 'supplier name', 'average_delay': 'average delay days'},
    color='average_delay',
    text_auto='.1f'
)

fig_delays.update_layout()
fig_delays.show()

## Supply Chain Analysis - Top Selling in last 30 days

In [14]:
fig_top_selling_30 = px.bar(
    top_selling_report_30_days,
    x='part_name',
    y='quantity',
    color='category',
    title='Top Selling Parts = last 30 days',
    labels={'part_name': 'Part Name', 'quantity': 'Quantity'},
    text_auto=True,
    hover_data=['category', 'quantity']
)

fig_top_selling_30.update_layout()
fig_top_selling_30.show()